# `mboostrs` — comprehensive family gallery

A **family** specifies the loss and response transformations used by a boosting model. It defines:

1. **Loss function** — what the algorithm minimises (e.g. squared error, deviance, pinball loss).
2. **Negative gradient** — the pseudo-residual that the base learners fit at each iteration.
3. **Response / link transforms** — converting between the linear predictor scale (`link`) and the response scale.

This notebook covers 18 families available in `mboostrs`, with examples of their response types and loss functions. Numerical parity with R is checked separately for specified inputs and tolerances.

Each section is self-contained. You can jump directly to the family you care about.

**Families covered:** Gaussian, Binomial, Poisson, NBinomial, GammaReg, GaussReg, Laplace, Huber, Quantile, Expectile, AdaExp, PropOdds, CoxPH, Weibull, Loglog, Lognormal, Multinomial, HingeLoss

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import altair as alt
import mboostrs as mb

from mboostrs.datasets import bodyfat, wpbc, iris

rng = np.random.default_rng(42)
print("mboostrs version:", mb.__version__)

## 1. Gaussian — squared-error regression

The **Gaussian** family (`Gaussian()`) minimises the squared error loss $\frac{1}{2}(y - f(x))^2$. This is the default family and corresponds to standard L2-boosting.

**When to use:** continuous unbounded responses with roughly symmetric errors — the classical normal-error regression setting.

**Data generating process:** $y = 1.5 \cdot x_1 - 0.8 \cdot x_2 + \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, 0.4^2)$.

The true coefficients are `x1=1.5, x2=-0.8`. With a learning rate of `nu=0.1`, the finite boosting path may not reach the unregularized least-squares coefficients.

In [ ]:
n = 500
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
y = 1.5 * x1 - 0.8 * x2 + rng.normal(0, 0.4, n)
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y})

m_gauss = mb.glmboost(
    "y ~ x1 + x2", data=data,
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_gauss.coefficients_}")
print(f"Risk at final iteration: {m_gauss.risk_[-1]:.4f}")

# Check recovery: true coefficients are [1.5, -0.8]
for name, true_val in [("x1", 1.5), ("x2", -0.8)]:
    est = m_gauss.coefficients_.get(name, 0)
    print(f"  {name}: estimated={est:.3f}, true={true_val:.1f}, diff={abs(est - true_val):.4f}")

## 2. Binomial — binary classification (logistic loss)

The **Binomial** family (`Binomial()`) minimises the logistic deviance (log-loss): $-\sum [y \log(p) + (1 - y) \log(1-p)]$. The additive predictor is on the log-odds scale; the inverse link is the logistic function.

**When to use:** binary outcomes where you want well-calibrated probabilities, not just hard classifications. Unlike SVM-style hinge loss, the Binomial family penalises incorrect predictions proportionally to their confidence.

**Data generating process:** $\eta = x_1 - 0.6 \cdot x_2$, $p = \sigma(\eta) = 1 / (1 + e^{-\eta})$, $y \sim \text{Bernoulli}(p)$.

In [ ]:
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
eta = 1.0 * x1 - 0.6 * x2
p = 1 / (1 + np.exp(-eta))
y_bin = (rng.uniform(size=n) < p).astype(float)
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y_bin})

m_bin = mb.glmboost(
    "y ~ x1 + x2", data=data, family=mb.Binomial(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
probs = m_bin.predict(type="response")
pred_class = (probs > 0.5).astype(int)
acc = (pred_class == y_bin).mean()
print(f"Classification accuracy: {acc:.3f}")
print(f"Mean predicted probability (class 1): {probs.mean():.3f}")
print(f"Base rate (class 1): {y_bin.mean():.3f}")

## 3. Poisson — count regression

The **Poisson** family (`Poisson()`) minimises the Poisson deviance $2\sum [y \log(y/\hat{\mu}) - (y - \hat{\mu})]$. The additive predictor is on the log scale; the inverse link is $\exp()$.

**When to use:** count data where the variance equals the mean (equidispersion). Typical examples: number of hospital visits, insurance claims, website clicks.

**Key concept — the offset:** `mboost` automatically sets an offset equal to $\log(\bar{y})$, which absorbs the constant term and any log-normal bias from random predictors. The intercept that `glmboost` fits is the *residual adjustment* on top of this offset.

In [ ]:
n = 400
x = rng.normal(size=n)
eta = 0.5 + 0.6 * x
y_cnt = rng.poisson(np.exp(eta))
data = pl.DataFrame({"x": x, "y": y_cnt.astype(float)})

m_pois = mb.glmboost(
    "y ~ x", data=data, family=mb.Poisson(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_pois.coefficients_}")
print(f"Mean count: {y_cnt.mean():.2f}, offset (log mean): {m_pois.offset_:.3f}")

## 4. NBinomial — overdispersed counts

The **NBinomial** family (`NBinomial()`) extends Poisson for overdispersed count data where the variance exceeds the mean. It introduces an additional dispersion parameter $\theta$ so that $\text{Var}(y) = \mu + \mu^2 / \theta$.

Use this family when count variation exceeds what a Poisson model represents. The `NBinomial` family estimates $\theta$ as a nuisance parameter during fitting.

**Data generating process:** overdispersion is introduced by mixing a Poisson rate with a Gamma random effect.

In [ ]:
n = 400
x = rng.gamma(2, 1, size=n)
mu = np.exp(0.4 + 0.5 * np.log(x))
# Generate negative-binomial-like overdispersed counts via Poisson-Gamma mixture
y_nb = rng.poisson(mu * rng.gamma(5, 1 / 5, size=n))
data = pl.DataFrame({"x": x, "y": y_nb.astype(float)})

m_nb = mb.glmboost(
    "y ~ x", data=data, family=mb.NBinomial(),
    control=mb.boost_control(mstop=250, nu=0.1),
)
print(f"Coefficients: {m_nb.coefficients_}")
print(f"Variance/mean ratio: {y_nb.var() / y_nb.mean():.2f}  (>1 indicates overdispersion)")

## 5. GammaReg — positive-skewed response

The **GammaReg** family (`GammaReg()`) uses the Gamma deviance and a log link. It's appropriate for continuous, strictly positive responses with constant coefficient of variation (standard deviation proportional to mean).

**When to use:** insurance claim sizes, rainfall amounts, waiting times — any positive-continuous data where the variance grows with the mean.

**Data generating process:** $y \sim \text{Gamma}(\text{shape}=3, \text{scale} = 0.5x / 3)$.

In [ ]:
n = 400
x = rng.uniform(0.5, 2, size=n)
shape = 3.0
scale = 0.5 * x / shape
y_gamma = rng.gamma(shape, scale)
data = pl.DataFrame({"x": x, "y": y_gamma})

m_gamma = mb.glmboost(
    "y ~ x", data=data, family=mb.GammaReg(),
    control=mb.boost_control(mstop=250, nu=0.1),
)
print(f"Coefficients: {m_gamma.coefficients_}")

## 6. GaussReg — Gaussian with identity link

**GaussReg** (`GaussReg()`) is the Gaussian family with an identity link — essentially standard L2 boosting without any response transformation. This differs from `Gaussian()` in that the latter uses the canonical identity link matching, while `GaussReg` is the classic GLM formulation.

**When to use:** standard regression where the response is already on the scale you want to model (no log or other link needed).

In [ ]:
n = 300
x = rng.uniform(-2, 2, size=n)
y = 2.0 + 0.8 * x + rng.normal(0, 0.5, n)
data = pl.DataFrame({"x": x, "y": y})

m_gr = mb.glmboost(
    "y ~ x", data=data, family=mb.GaussReg(),
    control=mb.boost_control(mstop=150, nu=0.1),
)
print(f"Coefficients: {m_gr.coefficients_}")

## 7. Laplace — L1 / median regression

The **Laplace** family (`Laplace()`) uses the absolute error loss $|y - f(x)|$, corresponding to L1 regression. It estimates the **conditional median** of $y$, not the mean.

**When to use:** heavy-tailed error distributions where outliers would pull the L2 (Gaussian) fit substantially. L1 regression is more robust — a single extreme outlier can push the L2 fit arbitrarily far, but the L1 median is resilient to up to 50% contamination.

**Data generating process:** $y = 1.2 \cdot x + \varepsilon$ with $\varepsilon \sim t_3$ (Student's t with 3 d.f., heavy-tailed).

In [ ]:
n = 400
x = rng.normal(size=n)
y = 1.2 * x + rng.standard_t(3, size=n)  # heavy-tailed errors
data = pl.DataFrame({"x": x, "y": y})

m_lap = mb.glmboost(
    "y ~ x", data=data, family=mb.Laplace(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_lap.coefficients_}")
print(f"Robust to outliers (Laplace = L1 loss)")

## 8. Huber — robust regression with automatic delta

The **Huber** family (`Huber()`) uses Huber's robust loss function, which is quadratic for small residuals and linear for large residuals. The transition point $\delta$ determines where the loss switches from L2 to L1.

Use Huber loss when small residuals should be treated quadratically and large residuals linearly. This limits the influence of extreme residuals while retaining a quadratic loss near zero.

**Data generating process:** $y = 1.2x + \varepsilon$ with every 10th observation contaminated by a large outlier $\sim \mathcal{N}(0, 5^2)$. This tests whether the Huber fit recovers the true slope despite contamination.

In [ ]:
n = 400
x = rng.normal(size=n)
y = 1.2 * x + rng.normal(0, 0.5, n)
y[::10] += rng.normal(0, 5, n // 10)  # add outliers every 10th obs
data = pl.DataFrame({"x": x, "y": y})

m_huber = mb.glmboost(
    "y ~ x", data=data, family=mb.Huber(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Coefficients: {m_huber.coefficients_}")
print(f"Huber delta parameter: {m_huber.coefficients_.get('_delta', 'auto')}")

## 9. Quantile — quantile regression (multiple taus)

The **Quantile** family (`Quantile(tau)`) uses the pinball (tilted absolute) loss: $\rho_{\tau}(r) = r \cdot (\tau - \mathbf{1}_{r < 0})$. It estimates the $\tau$-th conditional quantile of $y$.

Use quantile regression to estimate conditional quantiles, for example value-at-risk at $\tau=0.05$ or growth-chart quantiles at $\tau=0.10, 0.50, 0.90$.

Fitting several quantiles across $x$ shows how the conditional distribution changes. In the example below, the quantile curves spread as $|x|$ increases.

**Data generating process:** $y = 1.5x + (0.5 + 0.3|x|) \cdot \varepsilon$ — note the heteroscedastic errors.

In [ ]:
n = 600
x = rng.uniform(-2, 2, size=n)
y = 1.5 * x + (0.5 + 0.3 * np.abs(x)) * rng.normal(size=n)
data = pl.DataFrame({"x": x, "y": y})

taus = [0.1, 0.25, 0.5, 0.75, 0.9]
grid = np.linspace(x.min(), x.max(), 200)

for tau in taus:
    m_q = mb.gamboost(
        "y ~ bbs(x, df=4)", data=data, family=mb.Quantile(tau=tau),
        control=mb.boost_control(mstop=300, nu=0.1),
    )
    preds = m_q.predict(newdata=pl.DataFrame({"x": grid}), type="link")
    print(f"tau={tau:.1f}: base learner coefficient magnitude = {m_q.coef()}")
print("Quantile regression captures heteroscedasticity: spread increases with |x|")

In [ ]:
# Quantile fan plot: predicted quantiles across x
grid = np.linspace(x.min(), x.max(), 200)
data_grid = pl.DataFrame({"x": grid})

q_preds = []
for tau in taus:
    m_q = mb.gamboost(
        "y ~ bbs(x, df=4)", data=data, family=mb.Quantile(tau=tau),
        control=mb.boost_control(mstop=300, nu=0.1),
    )
    pred = m_q.predict(newdata=data_grid, type="link")
    q_preds.append(pl.DataFrame({"x": grid, "pred": pred, "tau": f"\u03c4={tau}"}))

q_df = pl.concat(q_preds).to_pandas()
data_pd = data.to_pandas()

alt.layer(
    alt.Chart(q_df).mark_line().encode(
        x="x", y="pred:Q", color="tau:N", strokeDash="tau:N"
    ).properties(title="Quantile fan plot"),
    alt.Chart(data_pd).mark_point(opacity=0.15, size=12).encode(
        x="x", y="y"
    ),
).interactive()


## 10. Expectile — expectile regression (asymmetric least squares)

**Expectiles** (`Expectile(tau)`) use an asymmetric squared loss, while quantiles use an asymmetric absolute loss. Expectile regression is an alternative when an asymmetric squared-error target is appropriate.

Use expectile regression when the conditional target is defined by asymmetric squared loss. At $\tau=0.5$, it estimates the conditional mean.

**Note:** $\tau=0.5$ gives the conditional mean (symmetric least squares), unlike quantile regression where $\tau=0.5$ gives the median.

In [ ]:
n = 400
x = rng.normal(size=n)
y = 0.8 * x + 0.3 * rng.normal(size=n)
data = pl.DataFrame({"x": x, "y": y})

for tau in [0.1, 0.5, 0.9]:
    m_e = mb.glmboost(
        "y ~ x", data=data, family=mb.Expectile(tau=tau),
        control=mb.boost_control(mstop=150, nu=0.1),
    )
    print(f"tau={tau:.1f}: slope={m_e.coefficients_.get('x', 0):.4f}")

## 11. AdaExp — AdaBoost exponential loss

The **AdaExp** family (`AdaExp()`) uses the exponential loss $\exp(-y \cdot f(x))$ from the original AdaBoost algorithm. The response must be coded as $\{-1, +1\}$.

The exponential loss places greater weight on misclassified observations, so label noise can have a large effect on the fit.

AdaBoost with stumps is equivalent to forward-stagewise additive modelling with exponential loss (Friedman, Hastie & Tibshirani, 2000).

In [ ]:
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
eta = 0.8 * x1 - 0.5 * x2
# AdaExp expects y in {-1, 1}
y_ada = np.sign(eta + rng.normal(0, 0.3, n))
y_ada[y_ada == 0] = 1
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y_ada})

m_ada = mb.glmboost(
    "y ~ x1 + x2", data=data, family=mb.AdaExp(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
probs_ada = m_ada.predict(type="response")
acc_ada = ((probs_ada > 0) == (y_ada > 0)).mean()
print(f"Accuracy: {acc_ada:.3f}")

## 12. PropOdds — proportional-odds ordinal regression

The **PropOdds** family (`PropOdds(n_levels=K)`) handles ordered categorical responses using the proportional-odds model: $\log\left(\frac{P(y \leq k)}{P(y > k)}\right) = \theta_k - f(x)$.

**When to use:** ordinal outcomes — survey ratings (1-5), disease severity stages, educational levels. The proportional-odds assumption means that the effect of predictors is the same across all thresholds (only the intercepts differ).

**Data generating process:** 3-level ordinal response with thresholds $\theta_1 = -0.5$, $\theta_2 = 1.0$.

In [ ]:
n = 400
x = rng.normal(size=n)
# 3-level ordinal response
eta_po = 0.8 * x
p1 = 1 / (1 + np.exp(-(-0.5 - eta_po)))
p2 = 1 / (1 + np.exp(-(1.0 - eta_po)))
u = rng.uniform(size=n)
y_ord = np.where(u < p1, 1, np.where(u < p2, 2, 3))
data = pl.DataFrame({"x": x, "y": y_ord})

m_po = mb.glmboost(
    "y ~ x", data=data, family=mb.PropOdds(n_levels=3),
    control=mb.boost_control(mstop=200, nu=0.1),
)
pred_po = m_po.predict(type="response")
print(f"PropOdds fitted: {pred_po.shape[1]} class probabilities")
print(f"First 5 rows of predicted probabilities:")
print(pred_po[:5])

## 13. CoxPH — Cox proportional hazards (survival)

The **CoxPH** family (`CoxPH()`) implements Cox's proportional hazards model via gradient boosting. The loss is the negative partial log-likelihood.

**When to use:** time-to-event (survival) data with right-censoring. The Cox model is semi-parametric — it makes no assumptions about the baseline hazard but assumes proportional hazards across covariate levels.

**Formula syntax:** `Surv(time, status) ~ x1 + x2` — following the same convention as R's `survival` package.

**Data generating process:** exponential survival times with censoring.

In [ ]:
n = 400
x = rng.normal(size=n)
eta_cox = 0.6 * x
t = np.exp(eta_cox + rng.gumbel(0, 1, size=n))
c = np.exp(rng.normal(2, 0.5, size=n))
time = np.minimum(t, c)
status = (t <= c).astype(float)
data = pl.DataFrame({"time": time, "status": status, "x": x})

m_cox = mb.glmboost(
    "Surv(time, status) ~ x", data=data, family=mb.CoxPH(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"CoxPH coefficients: {m_cox.coefficients_}")

## 14. Weibull — parametric Weibull survival

The **Weibull** family (`Weibull()`) is a fully parametric survival model using the Weibull distribution. Unlike CoxPH, it assumes a specific parametric form for the baseline hazard: $\lambda(t) = \alpha \kappa (\kappa t)^{\alpha - 1}$.

**When to use:** when the survival times follow a Weibull distribution — this can be checked via a log-log survival plot. Parametric models are more efficient than CoxPH when correctly specified.

**Data generating process:** Weibull survival times with shape=1.5 and scale depending on covariates.

In [ ]:
n = 400
x = rng.normal(size=n)
shape = 1.5
scale = np.exp(0.5 * x)
t = scale * rng.weibull(shape, size=n)
c = np.exp(rng.normal(2, 0.5, size=n))
time = np.minimum(t, c)
status = (t <= c).astype(float)
data = pl.DataFrame({"time": time, "status": status, "x": x})

m_wei = mb.glmboost(
    "Surv(time, status) ~ x", data=data, family=mb.Weibull(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Weibull coefficients: {m_wei.coefficients_}")

## 15. Loglog — log-logistic survival

The **Loglog** family (`Loglog()`) assumes a log-logistic distribution for survival times. The log-logistic distribution has heavier tails than the Weibull and can accommodate non-monotonic hazard functions.

**When to use:** when the hazard is non-monotonic (e.g., initially increasing then decreasing), which the Weibull cannot capture (its hazard is monotonic).

In [ ]:
n = 400
x = rng.normal(size=n)
eta_ll = 0.5 * x
t = np.exp(eta_ll + rng.logistic(0, 0.5, size=n))
c = np.exp(rng.normal(2, 0.5, size=n))
time = np.minimum(t, c)
status = (t <= c).astype(float)
data = pl.DataFrame({"time": time, "status": status, "x": x})

m_ll = mb.glmboost(
    "Surv(time, status) ~ x", data=data, family=mb.Loglog(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Loglog coefficients: {m_ll.coefficients_}")

## 16. Lognormal — log-normal survival

The **Lognormal** family (`Lognormal()`) assumes log-normal survival times — i.e., $\log(T)$ is normally distributed. This is a standard accelerated failure time (AFT) model.

**When to use:** when the log of survival time is approximately normal — a common pattern in many biological and industrial applications. The log-normal hazard increases to a peak then decreases (non-monotonic, like log-logistic).

In [ ]:
n = 400
x = rng.normal(size=n)
t = np.exp(0.5 * x + rng.normal(0, 0.5, size=n))
c = np.exp(rng.normal(2, 0.5, size=n))
time = np.minimum(t, c)
status = (t <= c).astype(float)
data = pl.DataFrame({"time": time, "status": status, "x": x})

m_ln = mb.glmboost(
    "Surv(time, status) ~ x", data=data, family=mb.Lognormal(),
    control=mb.boost_control(mstop=200, nu=0.1),
)
print(f"Lognormal coefficients: {m_ln.coefficients_}")

## 17. Multinomial — multi-class classification

The **Multinomial** family (`Multinomial(n_classes=K)`) handles $K$-class classification using the multinomial logit (softmax) model. It fits $K$ additive predictors — one per class — and normalises them via softmax to get class probabilities.

**When to use:** multi-class outcomes with 3+ unordered categories. Examples: species classification, digit recognition, disease subtyping.

**Data:** Fisher's Iris dataset — 3 species of iris, 4 predictors. We map species names to numeric labels 0, 1, 2.

In [ ]:
from sklearn.preprocessing import LabelEncoder

iris_df = iris()
data = iris_df
y_species = data["Species"].to_numpy()
le = LabelEncoder()
y_num = le.fit_transform(y_species)
data = data.with_columns(pl.Series("y", y_num))

m_multi = mb.gamboost(
    "y ~ bbs(Sepal.Length, df=4) + bbs(Sepal.Width, df=4) + bbs(Petal.Length, df=4) + bbs(Petal.Width, df=4)",
    data=data, family=mb.Multinomial(n_classes=3),
    control=mb.boost_control(mstop=100, nu=0.1),
)
pred_multi = m_multi.predict(type="response")
pred_class = pred_multi.argmax(axis=1)
acc_multi = (pred_class == y_num).mean()
print(f"Multinomial accuracy: {acc_multi:.3f}")
print(f"Class distribution: {np.bincount(y_num)}")

## 18. HingeLoss — SVM-like hinge loss

The **HingeLoss** family (`HingeLoss()`) uses the hinge loss $\max(0, 1 - y \cdot f(x))$, familiar from Support Vector Machines. The response must be coded as $\{-1, +1\}$.

**When to use:** binary classification with maximal-margin objectives. The hinge loss focuses on points near or on the wrong side of the decision boundary (support vectors), ignoring well-classified points completely.

**Comparison with Binomial:** Hinge loss is not strictly proper — it doesn't produce well-calibrated probabilities (it pushes for separation rather than probability estimation). Use `Binomial()` when you need probabilities; use `HingeLoss()` when you only care about the decision boundary.

In [ ]:
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
eta = 0.8 * x1 - 0.5 * x2
y_hinge = np.sign(eta + rng.normal(0, 0.2, n))
y_hinge[y_hinge == 0] = 1
data = pl.DataFrame({"x1": x1, "x2": x2, "y": y_hinge})

m_hinge = mb.glmboost(
    "y ~ x1 + x2", data=data, family=mb.HingeLoss(),
    control=mb.boost_control(mstop=150, nu=0.1),
)
pred_h = m_hinge.predict(type="response")
acc_h = ((pred_h > 0) == (y_hinge > 0)).mean()
print(f"Hinge accuracy: {acc_h:.3f}")

## Summary

The 18 families covered here apply to the following response types:

| Response type | Families |
|---|---|
| Continuous (symmetric) | Gaussian, GaussReg, Laplace, Huber |
| Continuous (positive) | GammaReg |
| Binary | Binomial, AdaExp, HingeLoss |
| Count | Poisson, NBinomial |
| Ordinal | PropOdds |
| Multi-class | Multinomial |
| Survival | CoxPH, Weibull, Loglog, Lognormal |
| Distributional (quantiles) | Quantile, Expectile |

The family determines the loss and response type; the boosting algorithm can therefore be applied to mean, quantile, ordinal, multiclass, and survival models.

In [ ]:
# Coefficient summary across families
coef_records = [
    {"family": "Gaussian", "x1": m_gauss.coefficients_.get("x1", 0), "x2": m_gauss.coefficients_.get("x2", 0)},
    {"family": "GaussReg", "x": m_gr.coefficients_.get("x", 0)},
    {"family": "Laplace", "x": m_lap.coefficients_.get("x", 0)},
    {"family": "Huber", "x": m_huber.coefficients_.get("x", 0)},
    {"family": "GammaReg", "x": m_gamma.coefficients_.get("x", 0)},
]
display(pl.DataFrame(coef_records))

# Risk paths for representative families
risk_data = []
for name, risk_arr in [("Gaussian", m_gauss.risk_), ("Binomial", m_bin.risk_),
                        ("Poisson", m_pois.risk_), ("Laplace", m_lap.risk_)]:
    for m, r in enumerate(risk_arr):
        risk_data.append({"iteration": m, "risk": r, "family": name})
risk_df = pl.DataFrame(risk_data)

alt.Chart(risk_df).mark_line().encode(
    x="iteration:Q", y="risk:Q", color="family:N",
    strokeDash="family:N",
).properties(title="Risk paths across families")


In [ ]:
# Coefficient comparison and risk paths across families
coef_records = [
    {"family": "Gaussian", "x1": m_gauss.coefficients_.get("x1", 0), "x2": m_gauss.coefficients_.get("x2", 0)},
    {"family": "GaussReg", "x": m_gr.coefficients_.get("x", 0)},
    {"family": "Laplace", "x": m_lap.coefficients_.get("x", 0)},
    {"family": "Huber", "x": m_huber.coefficients_.get("x", 0)},
    {"family": "GammaReg", "x": m_gamma.coefficients_.get("x", 0)},
]
display(pl.DataFrame(coef_records))

# Risk paths
risk_data = []
for name, risk_arr in [("Gaussian", m_gauss.risk_), ("Binomial", m_bin.risk_),
                        ("Poisson", m_pois.risk_), ("Laplace", m_lap.risk_)]:
    for m, r in enumerate(risk_arr):
        risk_data.append({"iteration": m, "risk": r, "family": name})
risk_df = pl.DataFrame(risk_data)

alt.Chart(risk_df).mark_line().encode(
    x="iteration:Q", y="risk:Q", color="family:N",
    strokeDash="family:N",
).properties(title="Risk paths across families")
